In [3]:
import string
import nltk
import numpy as np
import pandas as pd
from IPython.display import HTML, display
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize
from scipy.spatial.distance import cdist
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Download resource NLTK dasar
nltk.download('punkt')
nltk.download('stopwords')

# 2. Load Data (Sesuaikan path file Anda)
file_path = (
    r'C:\Python\pemrosesan teks\produk_oxylabs.csv'  # Contoh path
)
data = pd.read_csv(file_path)
print('Total rows : ', len(data))

# Inisialisasi PorterStemmer dan Stopwords
porter = PorterStemmer()
stop_words = set(stopwords.words('english'))


# Fungsi Preprocessing teks yang selaras
def preprocess_text(text):
  if not isinstance(text, str):
    return ''
  text = text.lower().translate(str.maketrans('', '', string.punctuation))
  tokens = word_tokenize(text)
  cleaned = [
      porter.stem(word)
      for word in tokens
      if word not in stop_words and len(word) > 2
  ]
  return ' '.join(cleaned)


# Jalankan preprocessing pada kolom deskripsi (asumsi nama kolom adalah 'Deskripsi')
data['Deskripsi_Clean'] = data['Deskripsi'].apply(preprocess_text)
corpus = data['Deskripsi_Clean'].to_list()

# =============================================================
# A. PENCARIAN DOKUMEN MENGGUNAKAN COSINE SIMILARITY (TF-IDF)
# =============================================================
print('\n--- PENCARIAN DENGAN COSINE SIMILARITY ---')

# Buat TF-IDF vectorizer (menggunakan seluruh corpus)
vectorizer_tfidf = TfidfVectorizer(sublinear_tf=True, max_features=5000)
documents_tfidf_features = vectorizer_tfidf.fit_transform(corpus)

# Definisikan Query
query = 'adventure game mario'

# Transformasi query ke ruang vektor TF-IDF
query_tfidf_features = vectorizer_tfidf.transform([query])

# Hitung Cosine Similarities
cosine_similarities = cosine_similarity(
    documents_tfidf_features, query_tfidf_features
)

TopK = 10
top_indices_cosine = cosine_similarities[:, 0].argsort()[::-1][:TopK]

# Kumpulkan hasil Cosine Similarity
titles_c, abstracts_c, scores_c = [], [], []
for doc_index in top_indices_cosine:
  # Asumsi kolom judul bernama 'Judul' atau 'Title', sesuaikan jika berbeda
  titles_c.append(
      data.iloc[doc_index].get('Judul', data.iloc[doc_index].get('Title', 'N/A'))
  )
  abstracts_c.append(
      data.iloc[doc_index]['Deskripsi'].replace('\n', ' ')
      if pd.notna(data.iloc[doc_index]['Deskripsi'])
      else ''
  )
  scores_c.append(cosine_similarities[doc_index, 0])

results_cosine = pd.DataFrame(
    {'Title': titles_c, 'Description': abstracts_c, 'Similarities': scores_c}
)
display(HTML(results_cosine.to_html()))


# =============================================================
# B. PENCARIAN DOKUMEN MENGGUNAKAN JACCARD SIMILARITY (BINARY)
# =============================================================
print('\n--- PENCARIAN DENGAN JACCARD SIMILARITY ---')

# Buat CountVectorizer biner (One-Hot Encoding) untuk korpus
vectorizer_binarized = CountVectorizer(
    binary=True, max_features=5000, vocabulary=vectorizer_tfidf.vocabulary_
)
documents_binarized_features = vectorizer_binarized.fit_transform(
    corpus
).toarray()

# Transformasi query ke bentuk biner
query_binarized_features = vectorizer_binarized.transform([query]).toarray()

# Hitung Jaccard Similarities menggunakan SciPy cdist (1 - jarak jaccard)
jaccard_similarities = 1 - cdist(
    documents_binarized_features, query_binarized_features, metric='jaccard'
)

top_indices_jaccard = jaccard_similarities[:, 0].argsort()[::-1][:TopK]

# Kumpulkan hasil Jaccard Similarity
titles_j, abstracts_j, scores_j = [], [], []
for doc_index in top_indices_jaccard:
  titles_j.append(
      data.iloc[doc_index].get('Judul', data.iloc[doc_index].get('Title', 'N/A'))
  )
  abstracts_j.append(
      data.iloc[doc_index]['Deskripsi'].replace('\n', ' ')
      if pd.notna(data.iloc[doc_index]['Deskripsi'])
      else ''
  )
  scores_j.append(jaccard_similarities[doc_index, 0])

results_jaccard = pd.DataFrame(
    {'Title': titles_j, 'Description': abstracts_j, 'Similarities': scores_j}
)
display(HTML(results_jaccard.to_html()))

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\JOSHUA\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\JOSHUA\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Total rows :  96

--- PENCARIAN DENGAN COSINE SIMILARITY ---


,Title,Description,Similarities
0,Super Mario 3D World,"Work together with your friends or compete for the crown in the first multiplayer 3D Mario game for the Wii U console. In the Super Mario 3D World game, players can choose to play as Mario, Luigi, Princess Peach or Toad.",0.410527
1,Super Mario Odyssey,"New Evolution of Mario Sandbox-Style Gameplay. Mario embarks on a new journey through unknown worlds, running and jumping through huge 3D worlds in the first sandbox-style Mario game since Super Mario 64 and Super Mario Sunshine. Set sail between expansive worlds aboard an airship, and perform all-new actions, such as throwing Mario's cap.",0.370458
2,Super Mario World: Super Mario Advance 2,"Mario hasn't even had a chance to savor his victory in the original Super Mario Advance, and already he's back for more in Super Mario World: Super Mario Advance 2. This game is a re-creation of the Super NES classic Super Mario World, featuring every last secret exit and all 74 levels of platform-jumping action. Once again, Princess Peach has been kidnapped by the ever-diabolical Bowser, and Mario and Luigi set out on a massive adventure to rescue her. As either Mario or Luigi, you'll solve puzzles, shoot fireballs, fly, and discover the hidden secrets in Dinosaur Island.",0.297951
3,Super Mario Advance 4: Super Mario Bros. 3,"A combination of exciting platforming action, wild power-ups, and charming Mario style has made Super Mario Bros. 3 a classic. With Super Mario Advance 4: Super Mario Bros. 3, you have your chance to partake in the special adventure anywhere you go on your Game Boy Advance. Your mission is to rescue the Mushroom Kingdom monarchs who have been transformed into animals by Bowser and his mischievous offspring, and in the process save the princess kidnapped by Bowser. Traverse eight huge worlds and encounter fun minigames and frantic boss battles as you explore the first Mario game to use an overworld map.",0.278244
4,Mario Kart Super Circuit,"Whoever hands out driver's licenses in the Mushroom Kingdom is about to be in some serious trouble. Classic Mario Kart action is back once again, this time in a colorful, lightning-fast game that you can take anywhere. Mario, Luigi, Peach, Toad, Wario, Yoshi, and Bowser are all tearing up the track, tossing Koopa Shells and laying banana peels along the way. Mario Kart: Super Circuit puts your driving skills to the test on 40 tracks, each littered with wild weapons and obstacles. Four players can compete in Mario Kart: Super Circuit using only one Game Pak. If each player has a copy of the game, bonus features and new modes of play become available.",0.241654
5,Super Mario 64,"Mario is super in a whole new way! Combining the finest 3-D graphics ever developed for a video game and an explosive sound track, Super Mario 64 becomes a new standard for video games. It's packed with bruising battles, daunting obstacle courses and underwater adventures. Retrieve the Power Stars from their hidden locations and confront your arch nemesis - Bowser, King of the Koopas!",0.237690
6,Super Mario Galaxy 2,"Super Mario Galaxy 2, the sequel to the galaxy-hopping original game, includes the gravity-defying, physics-based exploration from the first game, but is loaded with entirely new galaxies and features to challenge players. On some stages, Mario can pair up with his dinosaur buddy Yoshi and use his tongue to grab items and spit them back at enemies. Players can also have fun with new items such as a drill that lets our hero tunnel through solid rock. [Nintendo]",0.228778
7,Paper Mario,"Mario pals around in an all-new action adventure! Mario's back in his first adventure since Super Mario 64, and this time, Bowser's bent on preventing a storybook ending. When Princess Peach is kidnapped, Mario plots to rescue the seven Star Spirits and rid the Mushroom Kingdom of Koopa's cruel cohorts. As he travels from the tropical jungles of Lavalava Island to the frosty heights of Shiver Mountain, he'll 


--- PENCARIAN DENGAN JACCARD SIMILARITY ---


,Title,Description,Similarities
0,NFL 2K1,"In the end, NFL 2K1 is a deeper, more refined version of the original game.",0.111111
1,Gran Turismo 2,The world's most advanced racing game returns with thunderous new features.,0.111111
2,Super Mario 3D World,"Work together with your friends or compete for the crown in the first multiplayer 3D Mario game for the Wii U console. In the Super Mario 3D World game, players can choose to play as Mario, Luigi, Princess Peach or Toad.",0.100000
3,Super Mario Odyssey,"New Evolution of Mario Sandbox-Style Gameplay. Mario embarks on a new journey through unknown worlds, running and jumping through huge 3D worlds in the first sandbox-style Mario game since Super Mario 64 and Super Mario Sunshine. Set sail between expansive worlds aboard an airship, and perform all-new actions, such as throwing Mario's cap.",0.074074
4,NBA 2K1,"Visual Concepts has tweaked and buffed out the annoying gameplay traits that plagued its original hoops game, and the result is a highly polished product.",0.062500
5,Final Fantasy X,Final Fantasy's first game on the Playstation 2 is the first to use voice acting and a massive leap in the cinematic approach to storytelling. Stop reading and go play.,0.058824
6,Super Mario 64,"Mario is super in a whole new way! Combining the finest 3-D graphics ever developed for a video game and an explosive sound track, Super Mario 64 becomes a new standard for video games. It's packed with bruising battles, daunting obstacle courses and underwater adventures. Retrieve the Power Stars from their hidden locations and confront your arch nemesis - Bowser, King of the Koopas!",0.055556
7,Super Mario Galaxy 2,"Super Mario Galaxy 2, the sequel to the galaxy-hopping original game, includes the gravity-defying, physics-based exploration from the first game, but is loaded with entirely new galaxies and features to challenge players. On some stages, Mario can pair up with his dinosaur buddy Yoshi and use his tongue to grab items and spit them back at enemies. Players can also have fun with new items such as a drill that lets our hero tunnel through solid rock. [Nintendo]",0.051282
8,Resident Evil Code: Veronica,"Resident Evil Code: Veronica takes up the story of Claire Redfield, a character from Resident Evil 2. In that game she was searching for her brother Chris (a missing S.T.A.R.S. member) and in Resident Evil Code: Veronica we follow her to Europe where she continues her search at Umbrella headquarters.",0.045455
9,Super Mario World: Super Mario Advance 2,"Mario hasn't even had a chance to savor his victory in the original Super Mario Advance, and already he's back for more in Super Mario World: Super Mario Advance 2. This game is a re-creation of the Super NES classic Super Mario World, featuring every last secret exit and all 74 levels of platform-jumping action. Once again, Princess Peach has been kidnapped by the ever-diabolical Bowser, and Mario and Luigi set out on a massive adventure to rescue her. As either Mario or Luigi, you'll solve puzzles, shoot fireballs, fly, and discover the hidden secrets in Dinosaur Island.",0.043478
